# Stage 9 – Writing the report

**Goal:** write the final report in the structure of the course template, with every number read from the
output tables (so the text can never disagree with the results).

The text is split into one cell per template section, so each section is easy to find and edit. The last cells
build the Word, Markdown, HTML and PDF files and check the 5,000-word limit.

**Reads:** `outputs/tables/*`, `data/processed/*`, `outputs/figures/network_comparison.png`, `report/authors.json`
**Writes:** `report/report.docx`, `report/report.md`, `report/report.html`, and the PDF named after the authors

In [1]:
%run ./library/lib_00_settings.ipynb
%run ./library/lib_07_report_writer.ipynb
stage_log = get_stage_logger("09_report")

## Load the results

`ResultNumbers` loads the output tables once and offers small helpers that turn results into text, for example
`numbers.value("Test", "density", 3)` → `"0.263"`.

In [2]:
class ResultNumbers:
    """All results needed by the report, read from the pipeline outputs."""

    def __init__(self) -> None:
        summary_table = pd.read_csv(TABLES_FOLDER / "network_summary.csv").set_index("measure")
        self.summary = {format_name: summary_table[format_name] for format_name in FORMATS}
        self.node_measures = {f: read_table(TABLES_FOLDER / f"node_metrics_{FORMAT_FILE_TAG[f]}.csv") for f in FORMATS}
        self.edges = {f: read_table(PROCESSED_FOLDER / f"edges_{FORMAT_FILE_TAG[f]}.csv") for f in FORMATS}
        self.counts = pd.read_csv(TABLES_FOLDER / "dataset_counts.csv").set_index("format")
        self.file_audit = read_table(INTERIM_FOLDER / "file_audit.csv")
        self.observations = read_table(PROCESSED_FOLDER / "partnership_observations.csv")
        self.scorecard_checks = pd.read_csv(TABLES_FOLDER / "validation_scorecard_checks.csv")
        self.spot_checks = pd.read_csv(TABLES_FOLDER / "validation_spot_checks.csv")
        self.aggregation_checks = pd.read_csv(TABLES_FOLDER / "validation_aggregation_checks.csv")
        self.matches = read_table(PROCESSED_FOLDER / "match_metadata.csv")

    def value(self, format_name: str, measure: str, decimals: int = 2) -> str:
        """A network-level value as text (no decimals = thousands separator)."""
        number = float(self.summary[format_name][measure])
        return f"{number:,.0f}" if decimals == 0 else f"{number:.{decimals}f}"

    def three(self, measure: str, decimals: int = 2) -> str:
        """'a, b and c' for Test, ODI and T20I."""
        test_value, odi_value, t20_value = (self.value(f, measure, decimals) for f in FORMATS)
        return f"{test_value}, {odi_value} and {t20_value}"

    def table_row(self, label: str, measure: str, decimals: int = 2) -> list[str]:
        return [label] + [self.value(f, measure, decimals) for f in FORMATS]

    def leaders(self, format_name: str, measure: str, how_many: int = 3, decimals: int = 0) -> str:
        """The top batters on one measure, with their values."""
        top_rows = (self.node_measures[format_name].sort_values([measure, "player_id"], ascending=[False, True])
                    .head(how_many))
        return ", ".join(f"{name} ({value:,.{decimals}f})" for name, value in zip(top_rows["player_name"], top_rows[measure]))

    def heaviest_pairs(self, format_name: str, how_many: int = 2) -> str:
        top_edges = self.edges[format_name].nlargest(how_many, "weight")
        return "; ".join(f"{first}–{second} ({runs:,} runs in {count} partnerships)" for first, second, runs, count
                         in zip(top_edges.player_1_name, top_edges.player_2_name, top_edges.weight, top_edges.partnership_count))

    def community_members(self, format_name: str, community_number: int, how_many: int = 4) -> str:
        members = self.node_measures[format_name]
        members = members[members["community"] == community_number].nlargest(how_many, "strength")
        return ", ".join(members["player_name"])

    def community_profile(self, format_name: str, community_number: int) -> tuple[int, float]:
        """(number of players, median entry point) of one community."""
        members = self.node_measures[format_name]
        members = members[members["community"] == community_number]
        return len(members), float(members["median_entry_partnership"].median())

    def main_core(self, format_name: str) -> pd.DataFrame:
        measures = self.node_measures[format_name]
        return measures[measures["core_number"] == measures["core_number"].max()]


numbers = ResultNumbers()
report_details = json.loads((REPORT_FOLDER / "authors.json").read_text(encoding="utf-8"))

## Derived facts used in the text

In [3]:
total_files = len(numbers.file_audit)
included_files = int(numbers.file_audit["included"].sum())
total_observations = len(numbers.observations)
total_innings = int(numbers.counts["england_innings"].sum())
total_runs = int(numbers.counts["partnership_runs"].sum())
total_deliveries = int(numbers.counts["deliveries"].sum())
matches_without_england_batting = int((numbers.matches["n_innings_england"] == 0).sum())
first_match_day, last_match_day = numbers.matches["start_date"].min(), numbers.matches["start_date"].max()

players_per_format = [set(numbers.node_measures[f]["player_id"]) for f in FORMATS]
players_in_all_formats = len(players_per_format[0] & players_per_format[1] & players_per_format[2])
distinct_players = len(players_per_format[0] | players_per_format[1] | players_per_format[2])
zero_run_edges = sum(int((numbers.edges[f]["weight"] == 0).sum()) for f in FORMATS)

def rank_correlation(format_name: str, measure: str) -> float:
    measures = numbers.node_measures[format_name]
    return measures[measure].corr(measures["innings_batted"], method="spearman")

degree_vs_innings = {f: rank_correlation(f, "degree") for f in FORMATS}
strength_vs_innings = {f: rank_correlation(f, "strength") for f in FORMATS}
core_strength_share = {f: numbers.main_core(f)["strength"].sum() / numbers.node_measures[f]["strength"].sum() for f in FORMATS}
core_median_innings = {f: numbers.main_core(f)["innings_batted"].median() for f in FORMATS}
other_median_innings = {f: numbers.node_measures[f][numbers.node_measures[f]["core_number"]
                                                    < numbers.node_measures[f]["core_number"].max()]["innings_batted"].median()
                        for f in FORMATS}
outer_shell_batters = {f: int((numbers.node_measures[f]["core_number"] <= 2).sum()) for f in FORMATS}
community_profile = {(f, c): numbers.community_profile(f, c) for f in FORMATS
                     for c in sorted(numbers.node_measures[f]["community"].unique())}
scorecard_total, scorecard_passed = len(numbers.scorecard_checks), int(numbers.scorecard_checks["passed"].sum())

def percent(value: float) -> str:
    return f"{value:.0%}"

report_blocks: list[tuple] = []
def heading(text): report_blocks.append(("h", text))
def paragraph(text): report_blocks.append(("p", text))

## 1. Introduction

In [4]:
heading("Introduction")
paragraph(
    "Network analysis studies a system through the links between its parts. In team sport this idea fits well. "
    "Football passing networks are a common example. Cricket gets less attention, but batting has a natural "
    "link: the partnership. Two batters are always at the crease together. They build the score as a pair. When "
    "a wicket falls, one of them is replaced. Over many matches, these pairs form a network of who batted with "
    "whom and how many runs they added.")
paragraph(
    "In this project we study the batting partnerships of the England men's team from "
    f"{first_match_day:%d %B %Y} to {last_match_day:%d %B %Y}. England play three international formats. Test "
    "matches last up to five days and each side bats twice. One Day Internationals (ODIs) have 50 overs per "
    "side. Twenty20 Internationals (T20Is) have 20 overs per side. The formats differ in length, in tactics and "
    "often in players. So we build one network for each format and study all three with the same method.")
paragraph(
    "In each network, a node is a batter. An edge joins two batters who shared at least one partnership. The "
    "edge weight is the total number of runs the pair added together. The network shows how widely each "
    "batter is paired, where the runs are concentrated, which groups bat together often and who links "
    "groups that rarely meet. It does not measure how good a batter is, and we do not use it for that.")

## 2. Problem and Motivation

In [5]:
heading("Problem and Motivation")
paragraph(
    "Cricket statistics usually focus on single players, such as batting averages, or on lists of record "
    "partnerships. Neither shows the bigger picture of how a whole batting group is connected. A batting "
    "order is not a fixed set of pairs. Openers meet the middle order. The middle order meets the tail. "
    "Players come and go over the years. The three formats use different but overlapping squads. Looking at "
    "this structure tells us how much the batting depends on a small core, how far apart the top order and "
    "the tail are, and which players link one generation of the squad to the next. It is also a good test "
    "case for the measures we learned in the course, because we can apply the same design to three similar "
    "networks.")
paragraph(
    "We ask seven research questions (RQ). RQ1: What are the main structural features of the Test, ODI and "
    "T20I networks? RQ2: Which batters hold the most prominent positions in each format? RQ3: How are "
    "partnerships grouped, and what do cohesion and redundancy tell us? RQ4: Which batters have the highest "
    "weighted degree, that is, the most partnership runs? RQ5: Which communities of batters does a "
    "community-detection method find? RQ6: Which batters sit between other batters (betweenness)? RQ7: Is "
    "there a core-periphery structure?")
paragraph(
    "Our work adds three things. First, a checked and repeatable way to rebuild every partnership from "
    "ball-by-ball data. Second, three weighted networks, one per format. Third, a side-by-side comparison of "
    "the three networks using the same measures and the same settings. The study is descriptive. We do not "
    "predict results and we do not rank players by ability.")

## 3. Datasets

In [6]:
heading("Datasets")
paragraph(
    "Source. Our data come from Cricsheet (cricsheet.org). Cricsheet is a free, public collection of "
    "ball-by-ball cricket data. We used the file england_male_json.zip. It holds "
    f"{total_files} matches of England men's teams in JSON format (schema version 1.3.0). The file is dated "
    "11 November 2026. We keep it unchanged in our project folder, together with its checksum and a list of "
    "every file it contains. We did not type in any data by hand.")
paragraph(
    "Filters. We kept a match if it was a men's international match, England was playing, the format was "
    "Test, ODI or T20I, and the first day fell between 1 January 2020 and 29 October 2026. All "
    f"{total_files} files could be read. {included_files} matches passed the filters. The other "
    f"{total_files - included_files} were left out because they started before 2020. In "
    f"{matches_without_england_batting} limited-overs matches England never batted, because the game was "
    "abandoned or ended early. These matches give no partnerships. Table 1 shows the final counts.")
report_blocks.append(("table", "Table 1. The data after filtering (England batting innings only).",
    ["", "Test", "ODI", "T20I", "Total"],
    [[label] + [f"{int(numbers.counts.loc[f, column]):,}" for f in FORMATS] + [f"{int(numbers.counts[column].sum()):,}"]
     for label, column in [("Matches", "matches"), ("England innings", "england_innings"),
                           ("Deliveries", "deliveries"), ("Partnerships", "observations"),
                           ("Partnership runs", "partnership_runs")]]
    + [["Nodes (batters)"] + [numbers.value(f, "nodes", 0) for f in FORMATS] + [f"{distinct_players} different"],
       ["Edges (pairs)"] + [numbers.value(f, "edges", 0) for f in FORMATS] + ["-"]]))
paragraph(
    "Fields used. From each match we use the dates, gender, team type, match type and teams. We also use the "
    "player registry, which gives every player a fixed Cricsheet ID. From each England innings we read the "
    "deliveries in order. Each delivery names the striker and the non-striker. It gives the runs, where "
    "runs.total already includes extras, and it lists any wickets.")
paragraph(
    "Finding the partnerships. A partnership is the time two batters spend at the crease together. Our code "
    "reads the deliveries one by one and remembers the current pair. Each delivery adds its runs.total to "
    "that pair, once. When a wicket record names one of the two batters, the partnership ends after that "
    "ball. So the runs scored on the wicket ball still count for the old pair. We do not use a fixed list of "
    "dismissal types. If the pair changes without a wicket, we accept it only if the file records a batter "
    "replacement. Otherwise we log it as a problem. The last partnership closes at the end of the innings. "
    "If a pair splits up and later bats together again, for example after a retired-hurt batter comes back, "
    f"we count two partnerships. The result is a table with one row per partnership ({total_observations:,} "
    "rows). Each row has the match, date, innings, order in the innings, both player IDs, runs, number of "
    "balls, first and last ball and the reason the partnership ended.")
paragraph(
    "Building the networks. For each format we group the partnerships by pair. The edge weight is the sum of "
    "the pair's partnership runs. Each edge also stores the number of partnerships, matches and innings, and "
    "the first and last date. A player becomes a node only by taking part in a partnership. Squad members "
    f"who never batted with a partner are not nodes. Across the three formats there are {distinct_players} "
    f"different batters, and {players_in_all_formats} of them appear in all three networks.")
paragraph(
    "Tools. We did all data work and all measures in Python 3.13 with pandas 3.0 and NetworkX 3.4.2. The code "
    "is a set of Jupyter notebooks that run in a fixed order. Shared functions sit in library notebooks, and "
    "a test notebook checks them on small made-up matches with known answers. Each network is saved as a "
    "GraphML file for Gephi, the visualisation tool chosen for the project. The figures in this report were "
    "drawn in Python from the same GraphML files, using the ForceAtlas2 layout from NetworkX.")

## 4. Validity and Reliability

In [7]:
heading("Validity and Reliability")
paragraph(
    "Validity. Our network is a simple model of batting teamwork. An edge only means that two players batted "
    "together at least once. It does not mean friendship, talking to each other or helping each other score. "
    "Who bats with whom depends mostly on the batting order and on when wickets fall. So the network shows "
    "selection and batting position as much as real teamwork.")
paragraph(
    "The edge weight is the score the pair added while together. It includes extras such as wides, no-balls "
    "and byes, because these are part of runs.total. This is how cricket scorecards define a partnership. It "
    "is not the sum of the two batters' own runs. Weights add up over time, so players with more matches "
    f"collect more. A weight shows volume, not a scoring rate. {zero_run_edges} edges have a weight of zero: "
    "the pair batted together but added no runs. We keep them, because the link is real. One Test innings has "
    "five penalty runs that belong to the whole innings, not to any ball. We do not give them to any pair.")
paragraph(
    "Our filters define who is in the study: England men, three international formats and matches that "
    "started inside the time window. Cricsheet holds back seven England matches involving Afghanistan. Some "
    "of them may fall inside our window. In ODIs and T20Is many players were rotated, so some nodes rest on "
    "very few innings.")
paragraph(
    "Reliability. We never change the raw archive, and a checksum identifies it. The filters, dates, format "
    "names and the random seed (42) are set in one settings notebook. Players are identified by their "
    "Cricsheet ID, never by name. No player in our data appears under two spellings. The partnership rules "
    "are written down and tested on sixteen tricky cases, such as strike rotation, a run-out of the "
    "non-striker, extras, two wickets on one ball, a pair that bats together again, and penalty runs.")
paragraph(
    "We checked the partnership data in three separate ways. First, in all "
    f"{total_innings} innings the partnership runs add up to the runs of all deliveries ({total_runs:,} runs "
    f"from {total_deliveries:,} deliveries). Second, we added up each innings straight from the raw files, "
    "without our partnership code, and compared the totals with two facts Cricsheet records separately: the "
    f"target for the chasing side and the winning margin in runs. All {scorecard_passed} of {scorecard_total} "
    "comparisons agree. Matches decided by a rain rule were not checked. Third, we looked up "
    f"{len(numbers.spot_checks)} famous partnerships, such as Root and Brook's 454 at Multan in 2024 and "
    "Buttler and Hales' unbeaten 170 in the 2022 T20 World Cup semi-final. Our data match all of them "
    "exactly. The code raised no warnings: every change of pair is explained by a wicket, one retired-hurt "
    "batter, two recorded batter replacements or the end of the innings. After building the networks, "
    f"{len(numbers.aggregation_checks)} further checks hold. For example, the edge weights of each format add "
    "up to its partnership runs. Each GraphML file was also read back and compared with the original. The "
    "only random step is Louvain community detection. Its seed is fixed, and we report how much the result "
    "changes with other seeds.")

## 5. Measures and Results

In [8]:
heading("Measures and Results")
paragraph(
    "All three networks are undirected, weighted and one-mode. We follow one rule for weights. The weight is "
    "a tie strength: more runs means a stronger link. Shortest-path methods, however, read edge values as "
    "distances. So we use weights only in measures made for strengths: node strength, eigenvector centrality "
    "and modularity. In all path-based measures (betweenness, closeness and path length) every link counts "
    "as one step. Clustering, k-cores and redundancy also ignore weights. Table 2 sums up the networks. "
    "Figure 1 shows them.")
report_blocks.append(("table", "Table 2. Network-level measures. W = uses edge weights; all others ignore weights.",
    ["Measure", "Test", "ODI", "T20I"],
    [numbers.table_row("Nodes", "nodes", 0), numbers.table_row("Edges", "edges", 0),
     numbers.table_row("Density", "density", 3), numbers.table_row("Connected components", "components", 0),
     numbers.table_row("Mean degree", "degree_mean", 1), numbers.table_row("Maximum degree", "degree_max", 0),
     numbers.table_row("Degree centralisation", "degree_centralization", 3),
     numbers.table_row("Median edge weight (W)", "edge_weight_median", 1),
     numbers.table_row("Maximum edge weight (W)", "edge_weight_max", 0),
     numbers.table_row("Weight share of top 10% of edges (W)", "top10pct_edges_weight_share", 3),
     numbers.table_row("Edges with a single partnership (share)", "edges_single_observation_share", 3),
     numbers.table_row("Average local clustering", "avg_clustering", 3),
     numbers.table_row("Transitivity", "transitivity", 3),
     numbers.table_row("Average path length", "avg_path_length_lcc", 2),
     numbers.table_row("Diameter", "diameter_lcc", 0),
     numbers.table_row("Clustering, random graphs", "C_random", 3),
     numbers.table_row("Path length, random graphs", "L_random", 2),
     numbers.table_row("Small-world index sigma", "small_world_sigma", 2),
     numbers.table_row("Degree assortativity", "degree_assortativity", 3),
     numbers.table_row("Largest k of k-core", "max_core_k", 0),
     numbers.table_row("Nodes in main core", "max_core_nodes", 0),
     numbers.table_row("Louvain communities (W)", "louvain_communities", 0),
     numbers.table_row("Modularity Q (W)", "louvain_modularity", 3),
     numbers.table_row("Mean effective size", "mean_effective_size", 2),
     numbers.table_row("Mean constraint", "mean_constraint", 3)]))
report_blocks.append(("fig", FIGURES_FOLDER / "network_comparison.png",
    "Figure 1. The three batting-partnership networks, drawn the same way. Node size: strength (total "
    "partnership runs). Edge width: runs of the pair. Colour: Louvain community (numbered by size within each "
    "format; grey: fewer than three players). Layout: ForceAtlas2 on the unweighted links; positions only help "
    "the drawing and are not a measure."))

paragraph(
    "Size, density and connectedness (RQ1). Density is the share of all possible pairs that actually batted "
    "together, 2m/(n(n-1)). A component is a group of nodes linked by paths. These measures show how fully "
    f"the batters of a format are connected. Each network has about fifty batters ({numbers.three('nodes', 0)}). "
    "Each one is a single connected component. So any batter can reach any other through a chain of "
    "partnerships, and all measures use the whole network. Density is "
    f"{numbers.value('Test', 'density')} in Tests, {numbers.value('ODI', 'density')} in ODIs and "
    f"{numbers.value('T20I', 'density')} in T20Is. In other words, between a fifth and a quarter of all "
    "possible pairs batted together. On average, two batters are about two steps apart "
    f"({numbers.three('avg_path_length_lcc')}). The diameter, the longest of these shortest paths, is four in "
    "every format.")
paragraph(
    "Edge weights are very uneven. The typical (median) pair added "
    f"{numbers.three('edge_weight_median', 0)} runs. But the heaviest tenth of the edges holds "
    f"{percent(float(numbers.summary['Test']['top10pct_edges_weight_share']))} of all partnership runs in "
    "Tests and about half in ODIs and T20Is. Between 39% and 51% of the pairs batted together only once. The "
    f"heaviest pairs are, in Tests, {numbers.heaviest_pairs('Test')}; in ODIs, {numbers.heaviest_pairs('ODI')}; "
    f"in T20Is, {numbers.heaviest_pairs('T20I')}.")

paragraph(
    "Degree and strength (RQ2, RQ4). Degree is the number of different partners a batter had. It shows the "
    "breadth of a batter's links. Strength, or weighted degree, is the sum of the weights of a batter's "
    "edges. It is the total runs added in all the partnerships the batter took part in. We keep the two "
    "apart because they answer different questions: how many partners, and how many runs. Table 3 lists the "
    "leading batters.")
report_blocks.append(("table", "Table 3. Leading batters in each format. Betweenness and effective size ignore "
                               "weights; strength and eigenvector use them.",
    ["Measure", "Test", "ODI", "T20I"],
    [["Degree"] + [numbers.leaders(f, "degree") for f in FORMATS],
     ["Strength (runs)"] + [numbers.leaders(f, "strength") for f in FORMATS],
     ["Betweenness"] + [numbers.leaders(f, "betweenness", decimals=3) for f in FORMATS],
     ["Closeness"] + [numbers.leaders(f, "closeness_lcc", decimals=3) for f in FORMATS],
     ["Eigenvector"] + [numbers.leaders(f, "eigenvector_lcc", decimals=3) for f in FORMATS],
     ["Effective size"] + [numbers.leaders(f, "effective_size", decimals=1) for f in FORMATS]]))
test_measures = numbers.node_measures["Test"]
paragraph(
    f"In Tests, JE Root leads both lists by a wide margin. He batted with {test_measures['degree'].max()} of "
    f"the other {int(numbers.value('Test', 'nodes', 0)) - 1} batters. His strength is about twice that of "
    "the next player. This one hub explains why the Test network has the highest degree centralisation "
    f"({numbers.value('Test', 'degree_centralization')}, against {numbers.value('ODI', 'degree_centralization')} "
    f"in ODIs and {numbers.value('T20I', 'degree_centralization')} in T20Is). The index is 0 when all degrees "
    "are equal and 1 for a star. In ODIs, the highest degrees are shared by top-order and lower-order players. "
    "In T20Is, degree and strength split most clearly. LS Livingstone has the most partners, but JC Buttler has "
    "by far the highest strength, more than three times Livingstone's. Buttler's runs come from a few strong "
    "links. Livingstone has many partners, but each link is light.")
paragraph(
    "Both measures depend on opportunity. The rank correlation between degree and innings batted is "
    f"{degree_vs_innings['Test']:.2f}, {degree_vs_innings['ODI']:.2f} and {degree_vs_innings['T20I']:.2f}. "
    f"For strength it is {strength_vs_innings['Test']:.2f}, {strength_vs_innings['ODI']:.2f} and "
    f"{strength_vs_innings['T20I']:.2f}. So strength answers RQ4 in terms of involvement in partnership runs. "
    "It is not a measure of batting quality, since it also includes the partner's runs and extras. The degree "
    f"distributions (mean {numbers.three('degree_mean', 1)}; maximum {numbers.three('degree_max', 0)}) are "
    "skewed: many batters have few partners and a few have many. With only fifty nodes, however, we cannot "
    "test for a power law, so we make no claim that the networks are scale-free.")

paragraph(
    "Betweenness, closeness and eigenvector centrality (RQ2, RQ6). Betweenness is the share of shortest paths "
    "between other batters that pass through a given batter. Here it points to players who sit between parts "
    "of the squad that rarely meet. It does not mean those players control anything. Closeness is based on "
    "the average distance from a batter to everyone else. Eigenvector centrality, computed with weights, is "
    "high for batters with strong links to other well-linked batters. It converged in all formats.")
paragraph(
    "The three measures point to different players, most clearly in ODIs and T20Is. Eigenvector centrality "
    "follows the runs: Root, Pope and Brook in Tests; Root, Brook and Duckett in ODIs; Buttler, Salt and Brook "
    "in T20Is. All of them bat in the top or middle order. In Tests, Root also leads on betweenness "
    f"({test_measures['betweenness'].max():.3f}, more than twice the next value). In ODIs, however, the three "
    "highest betweenness values belong to AU Rashid, BA Carse and AAP Atkinson. In T20Is they belong to "
    "Livingstone, Rashid and CJ Jordan. Rashid, Carse, Atkinson and Jordan usually come in late (typically in "
    "the seventh to ninth partnership of an innings). They bat with whoever is left of the specialist batters "
    "and with the other bowlers. Rashid also played through the whole period. These players connect the "
    "lower order, which changes often, to the rest of the team. So high betweenness marks a bridge between "
    "the batting core and the tail, and between older and newer squads. It goes together with low strength.")

paragraph(
    "Clustering, redundancy and small-world structure (RQ3). A batter's local clustering coefficient is the "
    "share of the batter's partners who also batted with each other. Transitivity is the same idea for the "
    "whole network. A triangle means three players who have each batted with the other two. Average "
    f"clustering is {numbers.three('avg_clustering', 3)}. This is more than twice the value in random graphs "
    f"with the same number of nodes and edges ({numbers.three('C_random', 3)}; average of 200 random graphs). "
    "At the same time, paths are only 7% to 9% longer than in those random graphs. The small-world index, "
    "the clustering ratio divided by the path-length ratio, is about 2.2 in every format "
    f"({numbers.three('small_world_sigma')}). So the networks are tightly knit locally and still have short "
    "paths overall. Part of this is built into cricket: players in the same matches meet each other as "
    "wickets fall. With fifty nodes and a density above 0.2, short paths are almost certain anyway, so we "
    "treat this index as a rough guide only.")
paragraph(
    "Redundancy comes from Burt's idea of structural holes. A partner is redundant if that partner also "
    "batted with the batter's other partners. Effective size is degree minus this redundancy. Constraint is "
    "high when all of a batter's partners are linked to each other. On average, only about half of a "
    f"batter's partners are non-redundant (mean efficiency {numbers.three('mean_efficiency')}). This is the "
    "same high clustering seen from each batter's point of view. The batters with the largest effective size "
    "and lowest constraint have partners in groups that do not meet each other: Root, Stokes and Pope in "
    "Tests; Root, Rashid and SM Curran in ODIs; Livingstone, Rashid and MM Ali in T20Is.")

paragraph(
    "Communities (RQ5). A community is a group of nodes with more link weight inside the group than we would "
    "expect by chance. We used the Louvain method, which maximises weighted modularity Q (resolution 1, seed "
    "42). This is a computer method, and its groups are not official units of the team. Modularity is low to "
    f"moderate ({numbers.three('louvain_modularity', 3)}), so the groups overlap a lot. Over 100 different "
    f"seeds, the number of communities ranges from {numbers.value('Test', 'louvain_k_min', 0)} to "
    f"{numbers.value('Test', 'louvain_k_max', 0)} in Tests, {numbers.value('ODI', 'louvain_k_min', 0)} to "
    f"{numbers.value('ODI', 'louvain_k_max', 0)} in ODIs and {numbers.value('T20I', 'louvain_k_min', 0)} to "
    f"{numbers.value('T20I', 'louvain_k_max', 0)} in T20Is. We only discuss the larger groups and ignore groups "
    "of one or two players.")
paragraph(
    "Even so, the groups make sense. We describe them with two facts taken from the data: when a batter "
    "usually comes in (the number of the first partnership the batter joins in an innings, where 1 means "
    "opener) and the years the batter played. In Tests, the largest community "
    f"({community_profile[('Test', 1)][0]} players, such as {numbers.community_members('Test', 1)}) usually "
    f"comes in at partnership {community_profile[('Test', 1)][1]:.0f}. This is the lower-middle order and the "
    f"tail. The second ({community_profile[('Test', 2)][0]} players: {numbers.community_members('Test', 2)}) "
    f"comes in at partnership {community_profile[('Test', 2)][1]:.0f}. It is the middle order, which holds most "
    f"of the runs. Two small top-order groups (both coming in at partnership "
    f"{community_profile[('Test', 3)][1]:.0f}) are split by time, not by position. "
    f"{numbers.community_members('Test', 3, 3)} are the top three of the later years. "
    f"{numbers.community_members('Test', 4)} were the top order of 2020 and 2021. The ODI network also splits "
    f"by era. There is a top-order group most of whose members first appear in 2023 or later "
    f"({numbers.community_members('ODI', 2)}), the opening group of 2020 to 2023 "
    f"({numbers.community_members('ODI', 3)}), a middle-order group ({numbers.community_members('ODI', 4)}) "
    f"and a large lower-order group ({community_profile[('ODI', 1)][0]} players, usually coming in at "
    f"partnership {community_profile[('ODI', 1)][1]:.0f}). T20Is show the same three-way pattern: a current top "
    f"order ({numbers.community_members('T20I', 2)}), an earlier group of top- and middle-order batters "
    f"({numbers.community_members('T20I', 3)}) and a lower-order group of {community_profile[('T20I', 1)][0]} "
    "players. So the communities reflect batting position and squad generation.")

paragraph(
    "Core and periphery (RQ7). A k-core is the largest part of the network in which every node has at least "
    "k neighbours inside that part. A batter's core number is the highest k for which the batter is still "
    "inside. The main core is a group of batters who are all well linked to each other. The low shells form "
    f"the periphery. The main core is the {numbers.value('Test', 'max_core_k', 0)}-core in Tests "
    f"({numbers.value('Test', 'max_core_nodes', 0)} of {numbers.value('Test', 'nodes', 0)} batters), the "
    f"{numbers.value('ODI', 'max_core_k', 0)}-core in ODIs ({numbers.value('ODI', 'max_core_nodes', 0)} of "
    f"{numbers.value('ODI', 'nodes', 0)}) and the {numbers.value('T20I', 'max_core_k', 0)}-core in T20Is "
    f"({numbers.value('T20I', 'max_core_nodes', 0)} of {numbers.value('T20I', 'nodes', 0)}). The core holds "
    f"{percent(core_strength_share['Test'])}, {percent(core_strength_share['ODI'])} and "
    f"{percent(core_strength_share['T20I'])} of all node strength. Core batters played a median of "
    f"{core_median_innings['Test']:.0f}, {core_median_innings['ODI']:.0f} and {core_median_innings['T20I']:.0f} "
    f"innings, against {other_median_innings['Test']:.1f}, {other_median_innings['ODI']:.1f} and "
    f"{other_median_innings['T20I']:.1f} for the rest. {outer_shell_batters['Test']}, {outer_shell_batters['ODI']} "
    f"and {outer_shell_batters['T20I']} batters sit in the outer shells (core number 1 or 2). The Test core "
    "holds both specialist batters and long-serving bowlers: Root, Stokes and Brook, but also Broad, Anderson "
    "and Leach. On the other hand, the opening pair Crawley and Duckett, third and sixth in strength, are "
    "outside it. Openers bat mostly with each other and the number three. So they have heavy links but too "
    "few different partners. Core membership measures how embedded and long-serving a batter is, not how "
    "many runs the batter makes.")

paragraph(
    "Comparing the formats. We used the same definitions for all three networks, and they are of similar "
    "size. This makes the structural measures comparable. Raw counts and strengths are not comparable, "
    "because a Test side bats twice and for much longer "
    f"({int(numbers.counts.loc['Test', 'england_innings'])} innings and "
    f"{int(numbers.counts.loc['Test', 'deliveries']):,} deliveries, against "
    f"{int(numbers.counts.loc['T20I', 'england_innings'])} innings and "
    f"{int(numbers.counts.loc['T20I', 'deliveries']):,} deliveries in T20Is). Three differences stand out. "
    "The Test network is the densest, the most clustered and the most centralised. It is built around one "
    "batter who played through the whole period, and its runs sit on the fewest edges. The ODI network is "
    "the least dense and least centralised. It has the most one-off pairs, the highest modularity and the "
    "most non-redundant links. With fewer matches per player and a change of generation during the period, "
    "it is the most split by era. The T20I network has the lowest modularity and the sharpest core-periphery "
    f"divide: a core of {numbers.value('T20I', 'max_core_nodes', 0)} batters holds "
    f"{percent(core_strength_share['T20I'])} of the strength, surrounded by many players with very few innings.")

## 6. Conclusion

In [9]:
heading("Conclusion")
paragraph(
    "The three networks share a common shape. Each is one connected component. Any two batters are about two "
    "partnership steps apart. Between a fifth and a quarter of all possible pairs exist. Clustering is more "
    "than twice that of similar random graphs, and about half of each batter's links are redundant. This is "
    "what the mechanics of batting produce. An innings links its batters in a chain, and picking the same "
    "players again and again closes the triangles. On top of this, runs sit on a small number of links. So "
    "the weighted picture is much more uneven than the unweighted one.")
paragraph(
    "Who looks important depends on the measure, and the differences tell a story. Volume measures (strength "
    "and weighted eigenvector centrality) pick out the top-order and middle-order batters who pile up runs: "
    "Root in Tests and ODIs, and Buttler in T20Is. Breadth and bridging measures (degree, betweenness and "
    "effective size) pick out all-rounders and lower-order players in ODIs and T20Is, such as Rashid, Carse, "
    "Curran, Livingstone and Jordan. They bat with many different partners and link the changing tail to "
    "the batting core. Only in Tests does one player, Root, lead on every measure. That is why the Test "
    "network is the most centralised.")
paragraph(
    "The groups follow two things the method was never told about: batting position and time. Communities "
    "separate the lower, middle and top order, and they split the top order into different generations of "
    "openers. The k-core analysis shows a clear core and periphery in all formats. A core of about twenty "
    "batters holds two thirds of the strength in Tests and around nine tenths in ODIs and T20Is. The "
    "periphery is made of players with only a few innings. Heavy links and core membership are not the "
    "same thing. An established opening pair can have the heaviest edge in the network and still sit outside "
    "the main core.")
paragraph(
    "Across formats, Test cricket gives the most tightly knit and most hub-centred network. ODI cricket gives "
    "the most divided one. T20I cricket shows the biggest gap between a steady core and a wide, loosely "
    "linked periphery. These are statements about who batted with whom, not about ability or results.")

## 7. Critique

In [10]:
heading("Critique")
paragraph(
    "We think the project answers its descriptive questions for the group of matches it covers. The strongest "
    "part is how the networks were built. Every partnership can be traced back to single deliveries. Innings "
    "totals match separate scorecard facts. No partnership boundary was left unexplained. Our interpretation "
    "has more limits, for four reasons.")
paragraph(
    "First, opportunity drives most measures. Degree, strength, core number and betweenness all rise with the "
    "number of innings played (rank correlations above 0.9 for degree and strength). So the measures largely "
    "show who was picked most often. We reported innings next to the results, but we did not adjust the "
    "networks for it. Using runs per partnership as the weight, or comparing with random networks that "
    "shuffle partners inside each innings while keeping the batting order, would separate structure from "
    "playing time. Second, we put almost seven years into one static network. This mixes squads that never "
    "met, and the communities partly pick this up. Networks per year, or for a moving time window, would "
    "show the change directly. Third, the model is kept simple on purpose. Links have no direction and do "
    "not show which batter scored the runs. Extras are included in the weights. Path-based measures ignore "
    "weights, so a pair with one run counts the same as a pair with two thousand. A directed network of runs "
    "scored with each partner, or weighted path measures with a sound distance, could be tried instead. "
    "Fourth, community detection is unstable in dense networks of this size. Modularity is low and the number "
    "of groups changes with the seed, so only the broad pattern should be trusted.")
paragraph(
    "There are also limits in the data and in what we did. Matches held back by Cricsheet are missing. "
    "Batting role and handedness are not in the source, and we did not add them. The small-world and "
    "degree-distribution results are weak evidence with only fifty nodes. Our first proposal also asked "
    "whether the partnership structure differs between matches won and lost. We left that question out, and "
    "it remains future work, as does the same study for other national teams. Finally, the network figures "
    "in this report were drawn in Python from the exported GraphML files. They follow the style planned for "
    "Gephi, but they are not Gephi exports.")

## References

In [11]:
heading("References")
for reference in [
    "Blondel, V. D., Guillaume, J.-L., Lambiotte, R., Lefebvre, E. (2008). Fast unfolding of communities in "
    "large networks. Journal of Statistical Mechanics, P10008.",
    "Borgatti, S. P., Everett, M. G., Johnson, J. C. (2018). Analyzing Social Networks (2nd ed.). Sage.",
    "Burt, R. S. (1992). Structural Holes: The Social Structure of Competition. Harvard University Press.",
    "Cricsheet. Ball-by-ball cricket data, england_male_json.zip. https://cricsheet.org/downloads/ "
    "(archive dated 11 November 2026).",
    "Freeman, L. C. (1978). Centrality in social networks: conceptual clarification. Social Networks, 1(3), 215-239.",
    "Hagberg, A., Schult, D., Swart, P. (2008). Exploring network structure, dynamics, and function using "
    "NetworkX. Proceedings of SciPy 2008, 11-15.",
    "Jacomy, M., Venturini, T., Heymann, S., Bastian, M. (2014). ForceAtlas2, a continuous graph layout "
    "algorithm for handy network visualization designed for the Gephi software. PLoS ONE, 9(6), e98679.",
    "Newman, M. E. J. (2018). Networks (2nd ed.). Oxford University Press.",
    "Seidman, S. B. (1983). Network structure and minimum degree. Social Networks, 5(3), 269-287.",
    "Watts, D. J., Strogatz, S. H. (1998). Collective dynamics of 'small-world' networks. Nature, 393, 440-442.",
]:
    report_blocks.append(("ref", reference))

## Build the files and check the word limit

In [12]:
write_word_report(report_blocks, report_details, REPORT_FOLDER / "report.docx")
write_markdown_report(report_blocks, report_details, REPORT_FOLDER / "report.md")
write_html_report(report_blocks, report_details, REPORT_FOLDER / "report.html")

report_word_count = count_report_words(report_blocks, report_details)
stage_log.info("report written: %d words (limit %d) - %s", report_word_count, WORD_LIMIT,
               "OK" if report_word_count <= WORD_LIMIT else "TOO LONG")
assert report_word_count <= WORD_LIMIT, "The report is over the 5,000-word limit"

2026-11-16 20:50:43,874 INFO batnet.09_report: report written: 4913 words (limit 5000) - OK


### PDF

The course asks for the file name to be the group members' surnames in alphabetical order, joined by "and".

In [13]:
pdf_path = REPORT_FOLDER / f"{report_details['pdf_name']}.pdf"
print_html_to_pdf(REPORT_FOLDER / "report.html", pdf_path)
stage_log.info("wrote %s", pdf_path.name)

2026-11-16 20:50:45,470 INFO batnet.09_report: wrote Khan and Musale and Raja Mohamed and Salman.pdf
